# Practica 3 de introducción a la inteligencia artificial

## Entrenamiento de GAN

In [ ]:
# ============================
# DCGAN para figuras geométricas (Keras/TensorFlow)
# ============================
import os, io, glob, math, time, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import kagglehub

# ----------------------------
# 0) Config
# ----------------------------
IMG_SIZE   = 64          # 64x64 acelera y estabiliza la GAN
CHANNELS   = 3           # RGB
BATCH_SIZE = 32
Z_DIM      = 128         # dimensión del ruido
EPOCHS     = 100          # súbelo si quieres más calidad
OUT_DIR    = "gan_training"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# 1) Descarga dataset y localiza carpetas Circle/Square/Triangle
# ----------------------------
def download_dataset():
    print("Descargando dataset (kagglehub)…")
    base = kagglehub.dataset_download("dineshpiyasamara/geometric-shapes-dataset")
    print("Descargado en:", base)
    # Buscar la carpeta que contiene Circle/Square/Triangle
    for root, dirs, files in os.walk(base):
        if all(d in dirs for d in ["Circle", "Square", "Triangle"]):
            print("Carpeta de clases detectada:", root)
            return root
    raise RuntimeError("No se encontraron subcarpetas Circle/Square/Triangle.")

DATA_ROOT = download_dataset()

# ----------------------------
# 2) tf.data pipeline (→ [-1, 1] para tanh)
# ----------------------------
def decode_resize_norm(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], antialias=True)
    img = tf.cast(img, tf.float32) / 127.5 - 1.0   # [0,255] → [-1,1]
    return img

all_files = []
for cls in ["Circle", "Square", "Triangle"]:
    all_files.extend(glob.glob(os.path.join(DATA_ROOT, cls, "*.*")))
random.shuffle(all_files)

print(f"Total imágenes encontradas: {len(all_files)}")
ds = tf.data.Dataset.from_tensor_slices(all_files)
ds = ds.shuffle(buffer_size=len(all_files), reshuffle_each_iteration=True)
ds = ds.map(decode_resize_norm, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# ----------------------------
# 3) Modelos: Discriminador y Generador (DCGAN)
# ----------------------------
def build_discriminator(img_size=IMG_SIZE):
    inp = layers.Input(shape=(img_size, img_size, CHANNELS))
    x = layers.Conv2D(64, 4, strides=2, padding="same")(inp)           # 64→32
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(128, 4, strides=2, padding="same")(x)            # 32→16
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(256, 4, strides=2, padding="same")(x)            # 16→8
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(512, 4, strides=2, padding="same")(x)            # 8→4
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Flatten()(x)
    logits = layers.Dense(1)(x)  # from_logits=True en la loss
    return keras.Model(inp, logits, name="discriminator")

def build_generator(z_dim=Z_DIM, img_size=IMG_SIZE):
    # Arrancamos en 4x4 y vamos duplicando hasta 64x64
    init_size = img_size // 16  # 64→4
    inp = layers.Input(shape=(z_dim,))
    x = layers.Dense(init_size * init_size * 512, use_bias=False)(inp)
    x = layers.Reshape((init_size, init_size, 512))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(256, 4, strides=2, padding="same", use_bias=False)(x)  # 4→8
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False)(x)  # 8→16
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False)(x)   # 16→32
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(3, 4, strides=2, padding="same")(x)                    # 32→64
    out = layers.Activation("tanh")(x)  # salida en [-1,1]
    return keras.Model(inp, out, name="generator")

disc = build_discriminator()
gen  = build_generator()

# ----------------------------
# 4) Pérdidas y Optimizadores
# ----------------------------
bce = keras.losses.BinaryCrossentropy(from_logits=True)

def d_loss_fn(real_logits, fake_logits):
    # label smoothing ligero en reales
    real_labels = tf.ones_like(real_logits) * 0.9
    fake_labels = tf.zeros_like(fake_logits)
    real_loss = bce(real_labels, real_logits)
    fake_loss = bce(fake_labels, fake_logits)
    return real_loss + fake_loss

def g_loss_fn(fake_logits):
    fool_labels = tf.ones_like(fake_logits)  # el generador quiere que D diga "real"
    return bce(fool_labels, fake_logits)

G_OPT = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5, beta_2=0.999)
D_OPT = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5, beta_2=0.999)

# ----------------------------
# 5) Step de entrenamiento (tf.function)
# ----------------------------
@tf.function
def train_step(real_batch):
    bs = tf.shape(real_batch)[0]

    # a) Entrenar D
    z = tf.random.normal([bs, Z_DIM])
    with tf.GradientTape() as tape_d:
        fake_imgs = gen(z, training=True)
        real_logits = disc(real_batch, training=True)
        fake_logits = disc(fake_imgs, training=True)
        d_loss = d_loss_fn(real_logits, fake_logits)
    grads_d = tape_d.gradient(d_loss, disc.trainable_variables)
    D_OPT.apply_gradients(zip(grads_d, disc.trainable_variables))

    # b) Entrenar G
    z = tf.random.normal([bs, Z_DIM])
    with tf.GradientTape() as tape_g:
        fake_imgs = gen(z, training=True)
        fake_logits = disc(fake_imgs, training=True)
        g_loss = g_loss_fn(fake_logits)
    grads_g = tape_g.gradient(g_loss, gen.trainable_variables)
    G_OPT.apply_gradients(zip(grads_g, gen.trainable_variables))

    return d_loss, g_loss

# ----------------------------
# 6) Utilidades: muestreo y checkpoints
# ----------------------------
seed = tf.random.normal([25, Z_DIM])  # grid 5x5 fijo para evaluar progreso

def save_sample_grid(epoch, seed_noise=seed, out_dir=OUT_DIR):
    gen.eval = lambda *args, **kw: None  # no-op (compat)
    imgs = gen(seed_noise, training=False).numpy()
    imgs = (imgs + 1.0) / 2.0  # [-1,1]→[0,1]

    fig, axes = plt.subplots(5, 5, figsize=(5, 5))
    idx = 0
    for r in range(5):
        for c in range(5):
            axes[r, c].imshow(imgs[idx])
            axes[r, c].axis("off")
            idx += 1
    fig.tight_layout(pad=0)
    out_path = os.path.join(out_dir, f"epoch_{epoch:03d}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

ckpt = tf.train.Checkpoint(generator=gen, discriminator=disc,
                           g_opt=G_OPT, d_opt=D_OPT)
ckpt_manager = tf.train.CheckpointManager(ckpt, directory=os.path.join(OUT_DIR, "ckpts"), max_to_keep=3)

# ----------------------------
# 7) Entrenamiento
# ----------------------------
D_HISTORY = []
G_HISTORY = []

def plot_losses(epoch, out_dir=OUT_DIR):
    plt.figure(figsize=(6,4))
    plt.plot(D_HISTORY, label="D loss")
    plt.plot(G_HISTORY, label="G loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Evolución de pérdidas DCGAN")
    plt.legend()
    plt.tight_layout()

    out_path = os.path.join(out_dir, f"loss_epoch_{epoch:03d}.png")
    plt.savefig(out_path, dpi=150)
    print("Plot guardado en:", out_path)   # <── Añádelo
    plt.close()

print("Entrenando DCGAN…")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    d_losses, g_losses = [], []

    for real in ds:
        d_l, g_l = train_step(real)
        d_losses.append(float(d_l))
        g_losses.append(float(g_l))

    # Registrar pérdidas
    D_HISTORY.append(np.mean(d_losses))
    G_HISTORY.append(np.mean(g_losses))

    # Guardar grid de imágenes
    save_sample_grid(epoch)

    # Guardar gráfico de pérdidas
    plot_losses(epoch)

    # Checkpoints cada 5 epochs
    if epoch % 5 == 0:
        ckpt_manager.save()
        gen.save(os.path.join(OUT_DIR, f"generator_epoch_{epoch}.keras"))

    dt = time.time() - t0
    print(f"[{epoch:02d}/{EPOCHS}] D_loss={D_HISTORY[-1]:.4f} | G_loss={G_HISTORY[-1]:.4f} | {dt:.1f}s")


# ----------------------------
# 8) Función de inference: generar N imágenes (base64 opcional)
# ----------------------------
def sample_images(n=9, z_dim=Z_DIM):
    z = tf.random.normal([n, z_dim])
    imgs = gen(z, training=False).numpy()
    imgs = (imgs + 1.0) / 2.0
    fig, axes = plt.subplots(int(math.sqrt(n)), int(math.sqrt(n)), figsize=(4,4))
    idx = 0
    for r in range(axes.shape[0]):
        for c in range(axes.shape[1]):
            axes[r, c].imshow(imgs[idx])
            axes[r, c].axis("off")
            idx += 1
    plt.tight_layout(); plt.show()

#Ejemplo rápido de muestreo tras entrenar:
sample_images(9)


### GAN Generadora de imagenes

In [ ]:
import uuid
from PIL import Image

def gan_generate_images(n=1, z_dim=Z_DIM, out_dir="gan_outputs"):
    os.makedirs(out_dir, exist_ok=True)

    z = tf.random.normal([n, z_dim])
    imgs = gen(z, training=False).numpy()
    imgs = (imgs + 1.0) / 2.0  # [-1,1] → [0,1]

    output_paths = []

    for i in range(n):
        img = (imgs[i] * 255).astype(np.uint8)
        pil_img = Image.fromarray(img)
        filename = f"gan_{uuid.uuid4().hex}.png"
        full_path = os.path.join(out_dir, filename)
        pil_img.save(full_path)
        output_paths.append(full_path)

    return output_paths